# Context Engine Testing Notebook

This notebook tests the ContextEngine for ingestion and retrieval using sample files.

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname("__file__"), "../..")))

from context_engine.engine import ContextEngine
from context_engine.chunkers import get as get_chunker
from context_engine.loaders import get as get_loader
from context_engine.stores import get as get_store
from context_engine.embedders import get as get_embedder

/Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Setup

Configure the ContextEngine with default settings using HashEmbedder for local testing.

In [2]:
# Define the data directory
DATA_DIR = "/Users/nnandagopal/Desktop/personal_projects/RAG/data"
SAMPLE_PDF = os.path.join(DATA_DIR, "coffee_processing.pdf")

print(f"Data directory: {DATA_DIR}")
print(f"Sample PDF: {SAMPLE_PDF}")
print(f"PDF exists: {os.path.exists(SAMPLE_PDF)}")

Data directory: /Users/nnandagopal/Desktop/personal_projects/RAG/data
Sample PDF: /Users/nnandagopal/Desktop/personal_projects/RAG/data/coffee_processing.pdf
PDF exists: True


## Initialize ContextEngine

Create a ContextEngine instance with ChromaDB store and HashEmbedder for local testing.

In [3]:
# Initialize components
loader = get_loader("auto")
chunker = get_chunker("recursive")
store = get_store("chroma", embedder=get_embedder("openai"))

engine = ContextEngine(loader=loader, chunker=chunker, store=store)
print("ContextEngine initialized successfully")

ContextEngine initialized successfully


## Test 1: Parse Document

Test parsing a PDF document without chunking.

In [4]:
# Test 1: Ingest Document
doc_ids = await engine.ingest(SAMPLE_PDF)
print(f"Ingested {len(doc_ids)} documents")
print(f"Document IDs (first 5): {doc_ids[:5]}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 360/360 [00:00<00:00, 16113.17it/s]
[INFO] 2026-05-19 22:48:35,766 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-19 22:48:35,780 [RapidOCR] download_file.py:60: File exists and is valid: /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-19 22:48:35,780 [RapidOCR] main.py:57: Using /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-05-19 22:48:35,798 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-05-19 22:48:35,805 [RapidOCR] download_file.py:60: File exists and is valid: /Users/nnandagopal/Desktop/personal_projects/RAG/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-05-19 22:48:35,805 [RapidOCR] main.py:57: Using /Use

Ingested 22 documents
Document IDs (first 5): ['document:0', 'document:1', 'document:2', 'document:3', 'document:4']


In [5]:
# Test retrieval with a coffee processing query
query = "environment?"
results = await engine.retrieve(query, k=3)

print(f"Query: {query}")
print(f"Number of results: {len(results)}")

for i, result in enumerate(results):
    print('-'*50)
    print(f"Result {i+1}:")
    print(result.page_content, end='\n\n')
    print(f"Metadata: {result.metadata}")

Query: environment?
Number of results: 3
--------------------------------------------------
Result 1:
requires significant amounts of clean water and generates wastewater that must be properly managed to avoid environmental contamination.

Metadata: {'chunker': 'recursive', 'chunk_id': 12, 'source': 'document', 'distance': 0.6531180143356323}
--------------------------------------------------
Result 2:
Environmental Note: Modern wet mills are increasingly implementing water recycling systems and eco-pulping technologies to reduce water consumption by up to 90 percent while maintaining quality standards. ## Natural Processing Method

Metadata: {'chunker': 'recursive', 'source': 'document', 'chunk_id': 13, 'distance': 0.6984260082244873}
--------------------------------------------------
Result 3:
with optimal drying occurring in regions with consistent sunshine and low humidity.

Metadata: {'chunker': 'recursive', 'chunk_id': 15, 'source': 'document', 'distance': 0.7604600787162781}


In [6]:
# Test with another query
query2 = "effect of coffe processing on environment"
results2 = await engine.retrieve(query2, k=2)

print(f"Query: {query2}")
print(f"Number of results: {len(results2)}")

for i, result in enumerate(results2):
    print('-'*50)
    print(f"Result {i+1}:")
    print(result.page_content, end='\n\n')
    print(f"Metadata: {result.metadata}")

Query: effect of coffe processing on environment
Number of results: 2
--------------------------------------------------
Result 1:
Coffee processing is the method used to transform freshly picked coffee cherries into green coffee beans ready for roasting. The processing method significantly impacts the final flavor profile of the coffee. Different regions around the world have developed unique processing techniques based on their climate, available resources, and traditional practices. The choice of processing method can enhance certain flavor characteristics while suppressing others, making it one of the most critical

Metadata: {'source': 'document', 'chunker': 'recursive', 'chunk_id': 1, 'distance': 0.32509613037109375}
--------------------------------------------------
Result 2:
Processing begins immediately after harvest, as coffee cherries are highly perishable. The outer fruit must be removed, and the beans must be dried to prevent spoilage. The manner in which this is accomplis